#Data Access

In [0]:


spark.conf.set(
    "fs.azure.account.auth.type.nyctaxistorages.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.nyctaxistorages.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.nyctaxistorages.dfs.core.windows.net",
    "<YOUR_CLIENT_ID>"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.nyctaxistorages.dfs.core.windows.net",
    "<YOUR_CLIENT_SECRET>"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.nyctaxistorages.dfs.core.windows.net",
    "https://login.microsoftonline.com/<YOUR_TENANT_ID>/oauth2/token"
)

In [0]:
account_fqdn = "nyctaxistorages.dfs.core.windows.net"
placeholder_fqdn = "<storage-account>.dfs.core.windows.net"

spark.conf.set(f"fs.azure.account.oauth.provider.type.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth.provider.type.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.id.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.secret.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.endpoint.{placeholder_fqdn}"))

dbutils.fs.ls('abfss://silver@nyctaxistorages.dfs.core.windows.net/')

#Databases Creation

In [0]:
%sql
create database gold

#Data Reading and Writing and Creating delta tables

In [0]:
df_zone = spark.read.format('parquet')\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load('abfss://silver@nyctaxistorages.dfs.core.windows.net/trip_zone')

In [0]:
display(df_zone)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *

In [0]:
df_zone = df_zone.write.format('delta')\
                .mode('Append')\
                .option('path','abfss://gold@nyctaxistorages.dfs.core.windows.net/trip_zone')\
                .saveAsTable('gold.trip_zone')


In [0]:
%sql
select * from gold.trip_zone

###Trip Type

In [0]:
df_type = spark.read.format('parquet')\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load('abfss://silver@nyctaxistorages.dfs.core.windows.net/trip_type')

In [0]:
display(df_type)

In [0]:
gold= 'abfss://gold@nyctaxistorages.dfs.core.windows.net'

In [0]:
df_type.write.format('delta')\
                .mode('Append')\
                .save(f'{gold}/trip_type')


In [0]:
%sql
CREATE OR REPLACE TABLE gold.trip_type AS
SELECT *
FROM delta.`abfss://gold@nyctaxistorages.dfs.core.windows.net/trip_type`

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE gold.trip_type AS
    SELECT DISTINCT * FROM gold.trip_type
""")
display(spark.table("gold.trip_type"))

In [0]:
%sql
select * from gold.trip_type

###Trips 2023 data

In [0]:
df_trip = spark.read.format('parquet')\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load('abfss://silver@nyctaxistorages.dfs.core.windows.net/trip2023data')

In [0]:
df_trip.write.format('delta')\
                .mode('Append')\
                .save(f'{gold}/tripsdata')


In [0]:
%sql
CREATE OR REPLACE TABLE gold.tripsdata AS
SELECT *
FROM delta.`abfss://gold@nyctaxistorages.dfs.core.windows.net/tripsdata`

In [0]:
%sql
select * from gold.tripsdata